Bronze layer - ( raw data layer)

As i know the concept is to convert bronze means only the raw data without doing any modifications

In [0]:
df = spark.read.csv('/Workspace/Users/dehemiweerakoon@gmail.com/databricks-dubai-property-price/dubai_residential_data_2026.csv', header=True)


In [0]:
display(df.limit(5))

INSTANCE_DATE,PROCEDURE_EN,IS_FREE_HOLD_EN,AREA_EN,PROP_SB_TYPE_EN,TRANS_VALUE,ACTUAL_AREA,ROOMS_EN,NEAREST_METRO_EN,NEAREST_MALL_EN,NEAREST_LANDMARK_EN,PROJECT_EN,price_per_sqm,size_category,value_band
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2600000.0,140.95,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,BAHWAN TOWER,18446.26,Premium,High-End
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2731642.0,134.46,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,Imperial Avenue,20315.65,Premium,High-End
2026-03-09,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,620000.0,42.87,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,PANTHEON ELYSEE,14462.33,Compact,Entry
2026-03-06,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,540000.0,36.86,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,null,14650.03,Compact,Entry
2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,515000.0,66.5,1 B/R,Rashidiya Metro Station,City Centre Mirdif,null,null,7744.36,Mid-Size,Entry


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze_data_dubai_residental_data")

Silver step

1.  Remove duplicates
2. fix or standardize data types
3. handle missing/null values
4. filter out clearly bad records

In [0]:
# to convert this data first to sliver first load bronze and then fix
bronze_data = spark.read.table("bronze_data_dubai_residental_data")
display(bronze_data.limit(15))

INSTANCE_DATE,PROCEDURE_EN,IS_FREE_HOLD_EN,AREA_EN,PROP_SB_TYPE_EN,TRANS_VALUE,ACTUAL_AREA,ROOMS_EN,NEAREST_METRO_EN,NEAREST_MALL_EN,NEAREST_LANDMARK_EN,PROJECT_EN,price_per_sqm,size_category,value_band
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2600000.0,140.95,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,BAHWAN TOWER,18446.26,Premium,High-End
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2731642.0,134.46,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,Imperial Avenue,20315.65,Premium,High-End
2026-03-09,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,620000.0,42.87,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,PANTHEON ELYSEE,14462.33,Compact,Entry
2026-03-06,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,540000.0,36.86,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,null,14650.03,Compact,Entry
2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,515000.0,66.5,1 B/R,Rashidiya Metro Station,City Centre Mirdif,null,null,7744.36,Mid-Size,Entry
2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,342000.0,44.0,Studio,Rashidiya Metro Station,City Centre Mirdif,null,null,7772.73,Compact,Entry
2026-03-09,Sale,Free Hold,PALM JUMEIRAH,Flat,22000000.0,561.12,3 B/R,Jumeirah Beach Resdency,Marina Mall,Burj Al Arab,W Residences Dubai - The Palm,39207.3,Premium,Ultra-Premium
2026-03-06,Sale,Free Hold,BURJ KHALIFA,Hotel Apartment,1675000.0,85.35,1 B/R,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,THE DISTINCTION,19625.07,Mid-Size,High-End
2026-03-06,Sale,Free Hold,REMRAAM,Flat,1100000.0,93.4,2 B/R,null,null,Motor City,REMRAAM,11777.3,Spacious,Mid-Market
2026-03-06,Sale,Free Hold,DUBAI MARINA,Flat,1333500.0,41.37,Studio,Jumeirah Beach Resdency,Marina Mall,Burj Al Arab,BAY CENTRAL WEST & CENTRAL TOWERS,32233.5,Compact,Mid-Market


In [0]:
standard_column_names = []
for rec in bronze_data.columns:
    standard_column_names.append(rec.lower())
standard_column_names


['instance_date',
 'procedure_en',
 'is_free_hold_en',
 'area_en',
 'prop_sb_type_en',
 'trans_value',
 'actual_area',
 'rooms_en',
 'nearest_metro_en',
 'nearest_mall_en',
 'nearest_landmark_en',
 'project_en',
 'price_per_sqm',
 'size_category',
 'value_band']

In [0]:
silver_df = bronze_data.toDF(*standard_column_names)
display(silver_df.limit(5))

instance_date,procedure_en,is_free_hold_en,area_en,prop_sb_type_en,trans_value,actual_area,rooms_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,project_en,price_per_sqm,size_category,value_band
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2600000.0,140.95,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,BAHWAN TOWER,18446.26,Premium,High-End
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2731642.0,134.46,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,Imperial Avenue,20315.65,Premium,High-End
2026-03-09,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,620000.0,42.87,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,PANTHEON ELYSEE,14462.33,Compact,Entry
2026-03-06,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,540000.0,36.86,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,null,14650.03,Compact,Entry
2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,515000.0,66.5,1 B/R,Rashidiya Metro Station,City Centre Mirdif,null,null,7744.36,Mid-Size,Entry


In [0]:
# I first checked whether need to change the data type
silver_df.dtypes

[('instance_date', 'string'),
 ('procedure_en', 'string'),
 ('is_free_hold_en', 'string'),
 ('area_en', 'string'),
 ('prop_sb_type_en', 'string'),
 ('trans_value', 'string'),
 ('actual_area', 'string'),
 ('rooms_en', 'string'),
 ('nearest_metro_en', 'string'),
 ('nearest_mall_en', 'string'),
 ('nearest_landmark_en', 'string'),
 ('project_en', 'string'),
 ('price_per_sqm', 'string'),
 ('size_category', 'string'),
 ('value_band', 'string')]

from their i see the instance_date , trans_value , actual_area and the price_per_sqm is not in correct data format i convert as following

In [0]:
from pyspark.sql.functions import col
silver_df = silver_df.withColumn('instance_date',col('instance_date').cast('date'))
silver_df = silver_df.withColumn('trans_value',col('trans_value').cast('double'))
silver_df = silver_df.withColumn('actual_area',col('actual_area').cast('double'))
silver_df = silver_df.withColumn('price_per_sqm',col('price_per_sqm').cast('double'))
silver_df.dtypes

[('instance_date', 'date'),
 ('procedure_en', 'string'),
 ('is_free_hold_en', 'string'),
 ('area_en', 'string'),
 ('prop_sb_type_en', 'string'),
 ('trans_value', 'double'),
 ('actual_area', 'double'),
 ('rooms_en', 'string'),
 ('nearest_metro_en', 'string'),
 ('nearest_mall_en', 'string'),
 ('nearest_landmark_en', 'string'),
 ('project_en', 'string'),
 ('price_per_sqm', 'double'),
 ('size_category', 'string'),
 ('value_band', 'string')]

In [0]:
silver_df.summary().toPandas()

,summary,procedure_en,is_free_hold_en,area_en,prop_sb_type_en,trans_value,actual_area,rooms_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,project_en,price_per_sqm,size_category,value_band
0,count,18085,18085,18085,18085,18085,18085,17723,14513,14332,16433,14018,18085,18085,18085
1,mean,None,None,None,None,1857786.7325203214,94.38074371025695,None,None,None,None,5242.0,18644.97919435993,None,None
2,stddev,None,None,None,None,2756622.0506981793,77.23518450248275,None,None,None,None,0.0,20569.152879778107,None,None
3,min,Delayed Development,Free Hold,AL BARARI,Flat,2312.38,7.31,1 B/R,Airport Free Zone,City Centre Mirdif,Al Makhtoum International Airport,014 TOWER,33.1,Compact,Entry
4,25%,None,None,None,None,730000.0,54.74,None,None,None,None,5242.0,11399.83,None,None
5,50%,None,None,None,None,1190000.0,78.1,None,None,None,None,5242.0,15792.15,None,None
6,75%,None,None,None,None,2094015.0,115.49,None,None,None,None,5242.0,22586.58,None,None
7,max,Sell Development,Non Free Hold,Zaabeel Second,Workshop,1.0E8,4827.04,Studio,UAE Exchange Metro Station,Marina Mall,Sports City Swimming Academy,joya verde residences dubai,2207992.93,Spacious,Ultra-Premium


In [0]:
# drop the duplciate data
print(silver_df.count())
silver_df = silver_df.dropDuplicates()
print(silver_df.count())


18085
18085


In [0]:
from pyspark.sql.functions import col, sum

# Loop through every column and count how many values are null
null_counts = silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in silver_df.columns
])

# Display the results as a neat table in Databricks
display(null_counts)

instance_date,procedure_en,is_free_hold_en,area_en,prop_sb_type_en,trans_value,actual_area,rooms_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,project_en,price_per_sqm,size_category,value_band
0,0,0,0,0,0,0,362,3572,3753,1652,4067,0,0,0


In here all the null value columns are string format not the double or the integer

In [0]:
silver_df = silver_df.fillna("Unknown", subset=["rooms_en", "nearest_mall_en", "nearest_landmark_en","project_en","nearest_metro_en"])

In [0]:
silver_df.summary().toPandas()

,summary,procedure_en,is_free_hold_en,area_en,prop_sb_type_en,trans_value,actual_area,rooms_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,project_en,price_per_sqm,size_category,value_band
0,count,18085,18085,18085,18085,18085,18085,18085,18085,18085,18085,18085,18085,18085,18085
1,mean,None,None,None,None,1857786.7325203212,94.3807437102567,None,None,None,None,5242.0,18644.979194359836,None,None
2,stddev,None,None,None,None,2756622.050698179,77.23518450248281,None,None,None,None,0.0,20569.152879778132,None,None
3,min,Delayed Development,Free Hold,AL BARARI,Flat,2312.38,7.31,1 B/R,Airport Free Zone,City Centre Mirdif,Al Makhtoum International Airport,014 TOWER,33.1,Compact,Entry
4,25%,None,None,None,None,730000.0,54.74,None,None,None,None,5242.0,11399.83,None,None
5,50%,None,None,None,None,1190000.0,78.11,None,None,None,None,5242.0,15792.24,None,None
6,75%,None,None,None,None,2094015.0,115.49,None,None,None,None,5242.0,22586.58,None,None
7,max,Sell Development,Non Free Hold,Zaabeel Second,Workshop,1.0E8,4827.04,Unknown,Unknown,Unknown,Unknown,joya verde residences dubai,2207992.93,Spacious,Ultra-Premium


The number values are on  the range with noraml range so we dont need to fix those values

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_dubai_property")
display(silver_df.limit(10))

instance_date,procedure_en,is_free_hold_en,area_en,prop_sb_type_en,trans_value,actual_area,rooms_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,project_en,price_per_sqm,size_category,value_band
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2600000.0,140.95,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,BAHWAN TOWER,18446.26,Premium,High-End
2026-01-02,Sale,Free Hold,BURJ KHALIFA,Flat,2731642.0,134.46,2 B/R,Business Bay Metro Station,Dubai Mall,Downtown Dubai,Imperial Avenue,20315.65,Premium,High-End
2026-03-09,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,620000.0,42.87,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,PANTHEON ELYSEE,14462.33,Compact,Entry
2026-03-06,Sale,Free Hold,JUMEIRAH VILLAGE CIRCLE,Flat,540000.0,36.86,Studio,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,Unknown,14650.03,Compact,Entry
2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,515000.0,66.5,1 B/R,Rashidiya Metro Station,City Centre Mirdif,Unknown,Unknown,7744.36,Mid-Size,Entry
2026-03-06,Sale,Free Hold,INTERNATIONAL CITY PH 1,Flat,342000.0,44.0,Studio,Rashidiya Metro Station,City Centre Mirdif,Unknown,Unknown,7772.73,Compact,Entry
2026-03-09,Sale,Free Hold,PALM JUMEIRAH,Flat,2.2E7,561.12,3 B/R,Jumeirah Beach Resdency,Marina Mall,Burj Al Arab,W Residences Dubai - The Palm,39207.3,Premium,Ultra-Premium
2026-03-06,Sale,Free Hold,BURJ KHALIFA,Hotel Apartment,1675000.0,85.35,1 B/R,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,THE DISTINCTION,19625.07,Mid-Size,High-End
2026-03-06,Sale,Free Hold,REMRAAM,Flat,1100000.0,93.4,2 B/R,Unknown,Unknown,Motor City,REMRAAM,11777.3,Spacious,Mid-Market
2026-03-06,Sale,Free Hold,DUBAI MARINA,Flat,1333500.0,41.37,Studio,Jumeirah Beach Resdency,Marina Mall,Burj Al Arab,BAY CENTRAL WEST & CENTRAL TOWERS,32233.5,Compact,Mid-Market


In [0]:
from pyspark.sql.types import BooleanType
import pyspark.sql.functions as F
silver_df = silver_df.withColumn(
    "is_free_hold_en",
    F.when(F.lower(F.col("is_free_hold_en")).contains("Non Free Hold"), False)
     .otherwise(True).cast(BooleanType())
)

In here I convert the column data into a standard format, In here i have converted the data types of the columns into the 'date' and 'double' when the necessary columns found. After that we drop duplicates in here. then count the number of null values in each column. but we found that null values are in the string columns only . then i replaced the null value with default values. the data ranges are normal the price and the numbers are in the range so we dont need to correct them and convert them in here. In the last the is_free_hold_en convert into boolean value telling true and false

# Gold

Gold: Aggregate your Silver table into the summary view(s) needed to answer your problem 
statement (e.g. totals, averages, trends over time, top-N breakdowns by category). 

### 1.  Gold 1: Avg price / price-per-sqft by district (area_en)

In [0]:
import pyspark.sql.functions as F

gold_1_df = silver_df.groupBy("area_en").agg(
    F.avg("actual_area").alias("avg_actual_area"),
    F.avg("trans_value").alias("avg_trans_value"),
    F.avg("price_per_sqm").alias("avg_price_per_sqm"),
    F.count("trans_value").alias("count_trans_value")
).orderBy(F.col("avg_price_per_sqm").desc())
display(gold_1_df)

area_en,avg_actual_area,avg_trans_value,avg_price_per_sqm,count_trans_value
JUMEIRA BAY,302.58,2.965E7,103370.84,2
BLUEWATERS,171.51999999999998,1.1028231104166666E7,59474.33979166665,48
Zaabeel First,169.97625,9662500.0,56861.748750000006,16
PEARL JUMEIRA,172.245,8750000.0,50796.885,2
Trade Center First,118.5953846153846,4913284.615384615,48768.339230769234,13
THE WORLD,37.73,1700000.0,45056.98,1
Al Merkadh,39.95333333333334,1558200.0,42073.753333333334,3
DUBAI WATER CANAL,159.5428846153846,7022354.131538462,39418.07980769232,52
Jumeirah First,142.94608695652173,5402608.695652174,36735.36217391304,23
Marsa Dubai,156.75571428571428,6024385.8902857145,36711.584285714285,35


###  Gold 2: Avg price / volume by property type & room count

In [0]:
gold_2_df = silver_df.groupBy('prop_sb_type_en','rooms_en').agg(
    F.avg("trans_value").alias("avg_trans_value"),
    F.avg("price_per_sqm").alias("avg_price_per_sqm"),
    F.count("trans_value").alias("count_trans_value")
).orderBy(F.col("avg_price_per_sqm").desc())
display(gold_2_df)


prop_sb_type_en,rooms_en,avg_trans_value,avg_price_per_sqm,count_trans_value
Hotel Apartment,5 B/R,2.3E7,73473.04,1
Hotel Apartment,4 B/R,1.4375E7,53749.94,6
Flat,5 B/R,3.3054985083333332E7,52092.7875,12
Hotel Rooms,Studio,890172.1788679245,49938.59449685533,318
Hotel Apartment,3 B/R,8288127.5,42619.24324999999,40
Hotel Apartment,Unknown,2000000.0,41476.57,1
Hotel Rooms,1 B/R,1629973.6764705882,34969.894411764704,34
Hotel Apartment,2 B/R,4531404.223333334,33397.149185185175,135
Show Rooms,Unknown,4651828.0,32937.96,1
Shop,Unknown,2915328.338122271,31844.855065502205,229


### Gold 3: Monthly transaction volume & price trend

In [0]:
gold_3_df = silver_df.withColumn('year_month',F.date_format('instance_date','yyyy-MM')).groupBy('year_month').agg(
                F.sum("trans_value").alias("sum_trans_value"),
                
                F.avg("price_per_sqm").alias("avg_price_per_sqm"),
                F.sum("price_per_sqm").alias("sum_price_per_sqm"),
                F.avg('trans_value').alias('avg_trans_value'))
display(gold_3_df)

year_month,sum_trans_value,avg_price_per_sqm,sum_price_per_sqm,avg_trans_value
2026-01,7.731977073730001E9,18131.92066468254,7.310790412E7,1917653.0440798614
2026-03,4.39762191332E9,17963.65604564314,4.329241106999996E7,1824739.383120332
2026-02,8.176597520909999E9,18956.656240037493,8.086909551999995E7,1916689.5267018282
2026-04,4.406871810610001E9,19410.521847519187,4.811868366000006E7,1777681.246716418
2026-05,3.5780238949299994E9,19385.033151904987,3.917715199999998E7,1770422.5110984659
2026-06,4.659258337610002E9,18200.20204733727,4.6137512189999975E7,1837971.7308126239
2026-07,6.4772250652E8,18981.550204678366,6491690.170000001,1893925.4576608187


In [0]:
gold_1_df.write.format("delta").mode("overwrite").saveAsTable("gold_price_by_area")
gold_2_df.write.format("delta").mode("overwrite").saveAsTable("gold_price_by_type")
gold_3_df.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_view")